# AITR Lab02 — Exploration Notebook
**Knowledge Cutoff Effects on LLM-Generated Literature Reviews**  
Ankush Gupta | MRA2026003  

This notebook walks through:
1. Environment verification
2. Single-topic baseline generation (sanity check)
3. Single-topic RAG generation (sanity check)
4. Manual CAR verification on 5 citations
5. Preliminary results plotting (after full runs)

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import src/

from src.generator import load_model, generate_literature_review
from src.evaluator import extract_citations, compute_car
from src.rag_pipeline import rag_retrieve_and_format

print('Imports OK')

## 1. Load Model (run once)

In [ ]:
model, tokenizer = load_model(
    model_name='meta-llama/Llama-3.1-8B-Instruct',
    quantize=True
)
print('Model loaded.')

## 2. Baseline Generation — Pre-Cutoff Topic (Sanity Check)

In [ ]:
topic_pre = 'BERT and transformer-based text classification'
review_baseline = generate_literature_review(topic_pre, model, tokenizer)
print(review_baseline)

In [ ]:
# Extract and check citations
citations = extract_citations(review_baseline)
print('Citations found:', citations)

In [ ]:
car, statuses = compute_car(review_baseline)
print(f'CAR (pre-cutoff, baseline): {car:.3f}')
for c, s in statuses.items():
    print(f'  [{s}] {c}')

## 3. Baseline Generation — Post-Cutoff Topic (Expect Degradation)

In [ ]:
topic_post = 'LLM benchmark contamination detection 2025'
review_post = generate_literature_review(topic_post, model, tokenizer)
print(review_post)

In [ ]:
car_post, statuses_post = compute_car(review_post)
print(f'CAR (post-cutoff, baseline): {car_post:.3f}')
for c, s in statuses_post.items():
    print(f'  [{s}] {c}')

## 4. RAG Generation — Post-Cutoff Topic

In [ ]:
retrieved = rag_retrieve_and_format(topic_post, top_k=5)
print('--- Retrieved Papers ---')
print(retrieved)

In [ ]:
review_rag = generate_literature_review(topic_post, model, tokenizer, retrieved_papers=retrieved)
print(review_rag)

In [ ]:
car_rag, statuses_rag = compute_car(review_rag)
print(f'CAR (post-cutoff, RAG): {car_rag:.3f}')
for c, s in statuses_rag.items():
    print(f'  [{s}] {c}')

print(f'\nBaseline CAR: {car_post:.3f} → RAG CAR: {car_rag:.3f} (delta: +{car_rag-car_post:.3f})')

## 5. Plot Preliminary Results (after full runs)

In [ ]:
# Run after evaluate_baseline.py and evaluate_rag.py have completed
import subprocess
result = subprocess.run(
    ['python', '../scripts/plot_results.py', '--results_dir', '../results/'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)

In [ ]:
from IPython.display import Image
Image('../results/figures/car_barplot.png')

In [ ]:
Image('../results/figures/car_degradation.png')